# UFM-Transformer Kaggle Orchestration Notebook

This notebook orchestrates the project execution based on `instructions_integration.txt`.

✅ **Step 2 is skipped** (dataset download), because the dataset is assumed to already exist in the Kaggle environment.


## 0) Configuration (edit these paths first)

In [ ]:
from pathlib import Path
import subprocess
import sys
import shutil
import os

# =========================
# USER CONFIGURATION
# =========================
# Path where this repository is available in Kaggle (Working or Input).
# Example: Path('/kaggle/working/Ngbayafou_Deep Learning Biométrie Multimodale')
PROJECT_ROOT = Path('/kaggle/working/Ngbayafou_Deep Learning Biométrie Multimodale')

# Dataset is already present (Step 2 skipped).
# Update this to your Kaggle dataset location.
# Example: Path('/kaggle/input/socofing/data')
DATASET_PATH = Path('/kaggle/input/socofing')

# Outputs in Kaggle working directory
OUTPUT_DIR = Path('/kaggle/working/output')
RESULTS_DIR = Path('/kaggle/working/results')

# Training/eval parameters
EPOCHS = 50
BATCH_SIZE = 32
LR = 1e-4
IMAGE_SIZE = 224
NUM_WORKERS = 2
DEVICE = 'auto'  # auto, cuda, cpu

SRC_DIR = PROJECT_ROOT / 'src'
LATEX_DIR = PROJECT_ROOT / 'latex'

def run(cmd, cwd=None, check=True):
    print('\n' + '='*100)
    print('RUN:', ' '.join(str(x) for x in cmd))
    print('CWD:', cwd if cwd else os.getcwd())
    print('='*100)
    result = subprocess.run(cmd, cwd=cwd, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed with return code {result.returncode}: {cmd}')
    return result.returncode

print('PROJECT_ROOT:', PROJECT_ROOT)
print('SRC_DIR:', SRC_DIR)
print('DATASET_PATH:', DATASET_PATH)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('RESULTS_DIR:', RESULTS_DIR)


## 1) Analyze and verify project structure

In [ ]:
required_paths = [
    PROJECT_ROOT,
    SRC_DIR,
    SRC_DIR / 'main.py',
    SRC_DIR / 'requirements.txt',
    LATEX_DIR,
]

for p in required_paths:
    print(f'{p}:', 'OK' if p.exists() else 'MISSING')

print('\nTop-level content:')
for p in sorted(PROJECT_ROOT.iterdir()):
    print(' -', p.name)

print('\nSource files in src/:')
for p in sorted(SRC_DIR.glob('*.py')):
    print(' -', p.name)


## 2) Environment setup (Step 1 from the guide)

In [ ]:
# Install project dependencies
run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(SRC_DIR / 'requirements.txt')])

# Quick torch/device check
run([sys.executable, '-c', 'import torch; print(torch.__version__); print("CUDA available:", torch.cuda.is_available())'])


## 3) Dataset check (Step 2 skipped by request)

We only verify that the dataset path exists and contains files.

In [ ]:
if not DATASET_PATH.exists():
    raise FileNotFoundError(f'Dataset path not found: {DATASET_PATH}')

print('Dataset path exists:', DATASET_PATH)
sample_files = list(DATASET_PATH.rglob('*'))[:30]
print(f'Showing up to 30 entries ({len(sample_files)} found):')
for p in sample_files:
    print(' -', p)


## 4) Run training (Step 3)

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_cmd = [
    sys.executable, 'main.py',
    '--mode', 'train',
    '--dataset_path', str(DATASET_PATH),
    '--output_dir', str(OUTPUT_DIR),
    '--epochs', str(EPOCHS),
    '--batch_size', str(BATCH_SIZE),
    '--lr', str(LR),
    '--image_size', str(IMAGE_SIZE),
    '--num_workers', str(NUM_WORKERS),
    '--device', DEVICE,
]
run(train_cmd, cwd=str(SRC_DIR))


## 5) Run evaluation (Step 4)

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = OUTPUT_DIR / 'checkpoints' / 'best_model.pth'
if not MODEL_PATH.exists():
    raise FileNotFoundError(f'Expected best model not found: {MODEL_PATH}')

eval_cmd = [
    sys.executable, 'main.py',
    '--mode', 'eval',
    '--model_path', str(MODEL_PATH),
    '--dataset_path', str(DATASET_PATH),
    '--output_dir', str(RESULTS_DIR),
    '--batch_size', str(BATCH_SIZE),
    '--image_size', str(IMAGE_SIZE),
    '--num_workers', str(NUM_WORKERS),
    '--device', DEVICE,
]
run(eval_cmd, cwd=str(SRC_DIR))


## 6) Run visualization (Step 5)

In [ ]:
viz_cmd = [
    sys.executable, 'main.py',
    '--mode', 'visualize',
    '--model_path', str(MODEL_PATH),
    '--dataset_path', str(DATASET_PATH),
    '--output_dir', str(RESULTS_DIR),
    '--batch_size', str(BATCH_SIZE),
    '--image_size', str(IMAGE_SIZE),
    '--num_workers', str(NUM_WORKERS),
    '--device', DEVICE,
]
run(viz_cmd, cwd=str(SRC_DIR))


## 7) Copy generated files into LaTeX directory (Step 6)

In [ ]:
latex_figures = LATEX_DIR / 'figures'
latex_tables = LATEX_DIR / 'tables'
latex_figures.mkdir(parents=True, exist_ok=True)
latex_tables.mkdir(parents=True, exist_ok=True)

results_figures = RESULTS_DIR / 'figures'
results_tables = RESULTS_DIR / 'tables'

if results_figures.exists():
    for pdf in results_figures.glob('*.pdf'):
        shutil.copy2(pdf, latex_figures / pdf.name)

if results_tables.exists():
    for tex in results_tables.glob('*.tex'):
        shutil.copy2(tex, latex_tables / tex.name)

print('LaTeX figures:', [p.name for p in sorted(latex_figures.glob('*'))])
print('LaTeX tables:', [p.name for p in sorted(latex_tables.glob('*'))])


## 8) Placeholder tracking helper (Step 7 support)

This cell counts unreplaced `\textcolor{red}{...}` placeholders in `article.tex` and `memoire.tex`.

In [ ]:
def count_placeholders(tex_path: Path):
    text = tex_path.read_text(encoding='utf-8', errors='ignore')
    needle = 'textcolor{red}'
    return text.count(needle)

for name in ['article.tex', 'memoire.tex']:
    p = LATEX_DIR / name
    if p.exists():
        print(name, '-> placeholders:', count_placeholders(p))
    else:
        print(name, '-> not found')


## 9) Compile LaTeX PDFs (Step 8, optional on Kaggle)

If `pdflatex`/`bibtex` are not available in the Kaggle runtime, this step will fail and can be skipped.

In [ ]:
has_pdflatex = shutil.which('pdflatex') is not None
has_bibtex = shutil.which('bibtex') is not None
print('pdflatex available:', has_pdflatex)
print('bibtex available:', has_bibtex)

def compile_tex(doc_name: str):
    if not has_pdflatex:
        print(f'Skipping {doc_name}: pdflatex not available')
        return

    run(['pdflatex', f'{doc_name}.tex'], cwd=str(LATEX_DIR), check=False)
    if has_bibtex:
        run(['bibtex', doc_name], cwd=str(LATEX_DIR), check=False)
    run(['pdflatex', f'{doc_name}.tex'], cwd=str(LATEX_DIR), check=False)
    run(['pdflatex', f'{doc_name}.tex'], cwd=str(LATEX_DIR), check=False)

compile_tex('article')
compile_tex('memoire')


## 10) Final artifact summary

In [ ]:
def show_files(title, root: Path, patterns):
    print('\n' + title)
    if not root.exists():
        print('  (missing)')
        return
    found = []
    for pat in patterns:
        found.extend(root.glob(pat))
    found = sorted(set(found))
    if not found:
        print('  (none found)')
    else:
        for p in found:
            print(' -', p)

show_files('Model checkpoints', OUTPUT_DIR, ['**/*.pth'])
show_files('Evaluation/visualization outputs', RESULTS_DIR, ['**/*.json', '**/*.pdf', '**/*.tex'])
show_files('LaTeX artifacts', LATEX_DIR, ['*.tex', '*.pdf', 'figures/*.pdf', 'tables/*.tex'])
